# Mumbai Air Outlook: build the page's data in Colab

This notebook rebuilds `data.json` for the **Mumbai Air Outlook** page (an additional submission to the Google Mumbai Data Challenge, Zindi 2026) and shows the page inline.

Inputs you provide: the challenge files from the Zindi **Data** tab, `Train.csv`, `Test.csv`, `Stations.csv` and `AdditionalHistory.zip`. Everything else (the forecast pair, the page, the weather from Open-Meteo) comes from the repository and the Open-Meteo API.

Sources: challenge data (CPCB/CAAQMS via OpenAQ, XKDR, Rohan Rao/CPCB-derived; CC-BY-SA 4.0 via Zindi) and Open-Meteo (CC-BY 4.0, listed in the ClimateVerse challenge catalogue). No other source.

In [ ]:
!git clone -q https://github.com/vaibhavgupta9876/mumbai-air-outlook.git
%cd mumbai-air-outlook
import os, shutil, zipfile
os.makedirs('challenge', exist_ok=True)

In [ ]:
# Upload Train.csv, Test.csv, Stations.csv and AdditionalHistory.zip from the Zindi Data tab (about 20 MB in all).
from google.colab import files
up = files.upload()
for name in up:
    shutil.move(name, os.path.join('challenge', name))
if os.path.exists('challenge/AdditionalHistory.zip'):
    with zipfile.ZipFile('challenge/AdditionalHistory.zip') as z:
        z.extractall('challenge')
    # the archive may unpack into a folder of the same name or flat; make sure challenge/AdditionalHistory/ exists
    if not os.path.isdir('challenge/AdditionalHistory'):
        os.makedirs('challenge/AdditionalHistory', exist_ok=True)
        for f in os.listdir('challenge'):
            if f.startswith(('History_', 'Legacy_', 'Other_', 'XKDR_', 'Kaggle_', 'HistoryNotes')):
                shutil.move(os.path.join('challenge', f), os.path.join('challenge/AdditionalHistory', f))
print(sorted(os.listdir('challenge')), sorted(os.listdir('challenge/AdditionalHistory')))

In [ ]:
# Build data.json: challenge files + the forecast pair in forecast/ + live weather from Open-Meteo (about 30 s).
!python build_data.py --data challenge --pair forecast/low.csv forecast/high.csv --weather api --out data.json

In [ ]:
# A look at what was built: the Diwali table and the lens, drawn here with matplotlib (the page uses Google Charts).
import json
import pandas as pd
import matplotlib.pyplot as plt
d = json.load(open('data.json'))
print('Train ends', d['train_end_ist'], '| weather pull', d['weather_pull'], '| outlook', d['window'])
tbl = pd.DataFrame([{k: y[k] for k in ('year', 'date', 'n_stations', 'baseline', 'peak_rel', 'peak_value', 'clear_rel', 'excess')} for y in d['diwali']])
display(tbl)
fig, ax = plt.subplots(figsize=(11, 4))
for y in d['diwali']:
    ax.plot(y['rel'], [float('nan') if v is None else v for v in y['city']], lw=2 if y['n_stations'] >= 3 else 1,
            ls='-' if y['n_stations'] >= 3 else '--', label=f"{y['year']} ({y['n_stations']} stations)")
ax.set_xticks(range(-72, 144, 24)); ax.set_xticklabels(['D-3', 'D-2', 'D-1', 'Diwali', 'D+1', 'D+2', 'D+3', 'D+4', 'D+5'])
ax.axvline(0, color='k', lw=0.8); ax.set_ylabel('PM2.5 ug/m3 (median station)'); ax.legend(); ax.set_title('What Diwali does to Mumbai air')
plt.show()
print(d['diwali_note'])

In [ ]:
# Show the page itself, served from this runtime.
import subprocess, time
srv = subprocess.Popen(['python', '-m', 'http.server', '8000'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=900)

## Notes

- The forecast band is the pair of files we submit to the challenge (a five-model average scaled x0.95 and x1.4), not a statistical interval. The daily level is what a six-day forecast gets wrong most, so the band spans the level.
- Day labels use the night-time ventilation (boundary-layer height x wind, 20:00-07:00) against the 15 Oct-15 Dec 2025 terciles; a daily mean is dominated by the afternoon boundary layer and separates nothing.
- The Diwali city line is the median across stations with at least half the window covered and a window median above 10 ug/m3.
- Not an official advisory: for health decisions use CPCB and BMC bulletins.